In [2]:
!pip install experta
print('Uninstalling old frozendict and installing a compatible version...')
!pip install --upgrade frozendict


Uninstalling old frozendict and installing a compatible version...
  Attempting uninstall: frozendict
    Found existing installation: frozendict 1.2
    Uninstalling frozendict-1.2:
      Successfully uninstalled frozendict-1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
experta 1.9.4 requires frozendict==1.2, but you have frozendict 2.4.7 which is incompatible.


In [3]:
from experta import *

In [16]:
class Usuario(Fact):
    """Dados iniciais fornecidos pelo usuário"""
    objetivo = Field(str)
    biotipo = Field(str)
    restricao = Field(str)
    experiencia = Field(str)

class PerfilMetabolico(Fact):
    """Fato intermediário (Nível 1 de Encadeamento)"""
    tipo = Field(str)
    regras_disparadas = Field(tuple) # Alterado para tuple para compatibilidade com MATCH

class Planejamento(Fact):
    """Fato intermediário (Nível 2 de Encadeamento)"""
    dieta = Field(str)
    treino_base = Field(str)
    regras_disparadas = Field(tuple)

class RecomendacaoFinal(Fact):
    """Fato final (Nível 3 de Encadeamento)"""
    dieta_detalhe = Field(str)
    treino_detalhe = Field(str)
    suplementacao = Field(str)
    regras_disparadas = Field(tuple)

In [17]:
class MotorFitTech(KnowledgeEngine):

    # ---------------------------------------------------------
    # ESTRATÉGIA DE RESOLUÇÃO DE CONFLITO: SALIENCE
    # ---------------------------------------------------------
    @Rule(Usuario(objetivo="emagrecimento", biotipo="ectomorfo"), salience=10)
    def regra_1(self):
        """Regra 1: Alerta de Incompatibilidade Biológica"""
        self.declare(PerfilMetabolico(tipo="gasto_rapido", regras_disparadas=("Regra 1",)))

    # ---------------------------------------------------------
    # NÍVEL 1: Determinação do Perfil Metabólico
    # ---------------------------------------------------------
    @Rule(Usuario(objetivo="hipertrofia", biotipo="ectomorfo"))
    def regra_2(self):
        """Regra 2: Perfil Ectomorfo focado em Ganho"""
        self.declare(PerfilMetabolico(tipo="gasto_rapido", regras_disparadas=("Regra 2",)))

    @Rule(Usuario(objetivo="emagrecimento", biotipo="endomorfo"))
    def regra_3(self):
        """Regra 3: Perfil Endomorfo focado em Perda"""
        self.declare(PerfilMetabolico(tipo="acumulo_gordura", regras_disparadas=("Regra 3",)))

    # ---------------------------------------------------------
    # NÍVEL 2: Perfil Metabólico + Restrições -> Planejamento Base
    # ---------------------------------------------------------
    @Rule(PerfilMetabolico(tipo="gasto_rapido", regras_disparadas=MATCH.historico), Usuario(restricao="nenhuma"))
    def regra_4(self, historico):
        """Regra 4: Dieta Hipercalórica Padrão"""
        self.declare(Planejamento(
            dieta="hipercalorica_padrao",
            treino_base="forca_hipertrofia",
            regras_disparadas=historico + ("Regra 4",)
        ))

    @Rule(PerfilMetabolico(tipo="gasto_rapido", regras_disparadas=MATCH.historico), Usuario(restricao="intolerante_lactose"))
    def regra_5(self, historico):
        """Regra 5: Dieta Hipercalórica sem Lactose"""
        self.declare(Planejamento(
            dieta="hipercalorica_sem_lactose",
            treino_base="forca_hipertrofia",
            regras_disparadas=historico + ("Regra 5",)
        ))

    @Rule(PerfilMetabolico(tipo="acumulo_gordura", regras_disparadas=MATCH.historico))
    def regra_6(self, historico):
        """Regra 6: Dieta Hipocalórica e Cardio"""
        self.declare(Planejamento(
            dieta="hipocalorica_restrita",
            treino_base="deficit_calorico_com_cardio",
            regras_disparadas=historico + ("Regra 6",)
        ))

    # ---------------------------------------------------------
    # NÍVEL 3: Planejamento Base + Experiência -> Recomendações Finais
    # ---------------------------------------------------------
    @Rule(Planejamento(dieta="hipercalorica_padrao", treino_base="forca_hipertrofia", regras_disparadas=MATCH.historico),
          Usuario(experiencia="iniciante"))
    def regra_7(self, historico):
        """Regra 7: Finalização Ectomorfo Iniciante"""
        self.declare(RecomendacaoFinal(
            dieta_detalhe="Superávit de 500kcal com ingestão alta de leite integral e derivados.",
            treino_detalhe="Treino ABC 3x na semana, foco em exercícios compostos, descanso longo.",
            suplementacao="Hipercalórico e Creatina.",
            regras_disparadas=historico + ("Regra 7",)
        ))

    @Rule(Planejamento(dieta="hipercalorica_sem_lactose", treino_base="forca_hipertrofia", regras_disparadas=MATCH.historico),
          Usuario(experiencia="iniciante"))
    def regra_8(self, historico):
        """Regra 8: Finalização Ectomorfo com Restrição Iniciante"""
        self.declare(RecomendacaoFinal(
            dieta_detalhe="Superávit de 500kcal utilizando leite de amêndoas, aveia e carnes magras.",
            treino_detalhe="Treino ABC 3x na semana, foco em execução correta.",
            suplementacao="Proteína isolada da carne (Beef Protein) e Creatina.",
            regras_disparadas=historico + ("Regra 8",)
        ))

    @Rule(Planejamento(dieta="hipocalorica_restrita", treino_base="deficit_calorico_com_cardio", regras_disparadas=MATCH.historico),
          Usuario(experiencia="iniciante"))
    def regra_9(self, historico):
        """Regra 9: Finalização Endomorfo Iniciante"""
        self.declare(RecomendacaoFinal(
            dieta_detalhe="Déficit de 400kcal. Alto teor de proteínas e redução de carbo simples.",
            treino_detalhe="Treino AB 4x na semana + 20 min de cardio moderado pós-treino.",
            suplementacao="Whey Protein e Café puro.",
            regras_disparadas=historico + ("Regra 9",)
        ))

    @Rule(Planejamento(dieta="hipocalorica_restrita", treino_base="deficit_calorico_com_cardio", regras_disparadas=MATCH.historico),
          Usuario(experiencia="avancado"))
    def regra_10(self, historico):
        """Regra 10: Finalização Endomorfo Avançado"""
        self.declare(RecomendacaoFinal(
            dieta_detalhe="Déficit de 600kcal com estratégia de ciclagem de carboidratos.",
            treino_detalhe="Treino ABCDE de alta intensidade (Drop-sets) + 40 min HIIT alternados.",
            suplementacao="Whey Isolado, Termogênico e BCAA.",
            regras_disparadas=historico + ("Regra 10",)
        ))

    # EXPLICABILIDADE EXIGIDA PELA RUBRICA
    @Rule(RecomendacaoFinal(dieta_detalhe=MATCH.d, treino_detalhe=MATCH.t, suplementacao=MATCH.s, regras_disparadas=MATCH.h), salience=-10)
    def imprimir_resultado(self, d, t, s, h):
        print("\n" + "="*60)
        print("🎯 RECOMENDAÇÕES GERADAS PELO SISTEMA ESPECIALISTA")
        print("="*60)
        print(f"🥗 DIETA:       {d}")
        print(f"🏋️‍♂️ TREINO:      {t}")
        print(f"💊 SUPLEMENTOS: {s}")
        print("-"*60)
        print(f"🔍 EXPLICABILIDADE: Esta decisão foi tomada porque as regras {', '.join(h)} dispararam em cadeia.")
        print("="*60 + "\n")

In [18]:
def executar_teste(titulo, dados_usuario):
    print(f"\n🚀 {titulo}")
    engine = MotorFitTech()
    engine.reset()
    engine.declare(Usuario(**dados_usuario))
    engine.run()

In [19]:
"""
SAÍDA ESPERADA PARA O CASO 1:
- Dieta: Superávit de 500kcal com ingestão alta de leite integral e derivados.
- Treino: Treino ABC 3x na semana, foco em exercícios compostos, descanso longo.
- Suplementos: Hipercalórico e Creatina.
- Explicabilidade: Regra 2 -> Regra 4 -> Regra 7 dispararam.
"""
executar_teste("Caso de Teste 1: Ectomorfo Iniciante Padrão", {
    "objetivo": "hipertrofia", "biotipo": "ectomorfo", "restricao": "nenhuma", "experiencia": "iniciante"
})


🚀 Caso de Teste 1: Ectomorfo Iniciante Padrão

🎯 RECOMENDAÇÕES GERADAS PELO SISTEMA ESPECIALISTA
🥗 DIETA:       Superávit de 500kcal com ingestão alta de leite integral e derivados.
🏋️‍♂️ TREINO:      Treino ABC 3x na semana, foco em exercícios compostos, descanso longo.
💊 SUPLEMENTOS: Hipercalórico e Creatina.
------------------------------------------------------------
🔍 EXPLICABILIDADE: Esta decisão foi tomada porque as regras Regra 2, Regra 4, Regra 7 dispararam em cadeia.



In [20]:
"""
SAÍDA ESPERADA PARA O CASO 2:
- Dieta: Superávit de 500kcal utilizando leite de amêndoas, aveia e carnes magras.
- Treino: Treino ABC 3x na semana, foco em execução correta.
- Suplementos: Proteína isolada da carne (Beef Protein) e Creatina.
- Explicabilidade: Regra 2 -> Regra 5 -> Regra 8 dispararam.
"""
executar_teste("Caso de Teste 2: Ectomorfo com Restrição Alimentar", {
    "objetivo": "hipertrofia", "biotipo": "ectomorfo", "restricao": "intolerante_lactose", "experiencia": "iniciante"
})


🚀 Caso de Teste 2: Ectomorfo com Restrição Alimentar

🎯 RECOMENDAÇÕES GERADAS PELO SISTEMA ESPECIALISTA
🥗 DIETA:       Superávit de 500kcal utilizando leite de amêndoas, aveia e carnes magras.
🏋️‍♂️ TREINO:      Treino ABC 3x na semana, foco em execução correta.
💊 SUPLEMENTOS: Proteína isolada da carne (Beef Protein) e Creatina.
------------------------------------------------------------
🔍 EXPLICABILIDADE: Esta decisão foi tomada porque as regras Regra 2, Regra 5, Regra 8 dispararam em cadeia.



In [21]:
"""
SAÍDA ESPERADA PARA O CASO 3:
- Dieta: Déficit de 600kcal com estratégia de ciclagem de carboidratos.
- Treino: Treino ABCDE de alta intensidade (Drop-sets) + 40 min HIIT alternados.
- Suplementos: Whey Isolado, Termogênico e BCAA.
- Explicabilidade: Regra 3 -> Regra 6 -> Regra 10 dispararam.
"""
executar_teste("Caso de Teste 3: Endomorfo Avançado para Definição", {
    "objetivo": "emagrecimento", "biotipo": "endomorfo", "restricao": "nenhuma", "experiencia": "avancado"
})


🚀 Caso de Teste 3: Endomorfo Avançado para Definição

🎯 RECOMENDAÇÕES GERADAS PELO SISTEMA ESPECIALISTA
🥗 DIETA:       Déficit de 600kcal com estratégia de ciclagem de carboidratos.
🏋️‍♂️ TREINO:      Treino ABCDE de alta intensidade (Drop-sets) + 40 min HIIT alternados.
💊 SUPLEMENTOS: Whey Isolado, Termogênico e BCAA.
------------------------------------------------------------
🔍 EXPLICABILIDADE: Esta decisão foi tomada porque as regras Regra 3, Regra 6, Regra 10 dispararam em cadeia.

